<a href="https://colab.research.google.com/github/Fodhe-bot/Beginners-into-python-and-JSON/blob/main/Extract_Key_Dates_from_SDF_Documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf pypdf2 pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 101.0 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving sample-sdf-document.pdf to sample-sdf-document.pdf


In [5]:
import fitz

In [6]:
import json

In [7]:
pdf_path = "sample-sdf-document.pdf"
doc = fitz.open(pdf_path)

In [3]:
extracted_data = {
    "basic_and_dates": [],
    "table_data": []
}

In [23]:
# --- PART 1: EXTRACTING BASIC INFO & DATES (Pages 2 & 3) ---
for page_num in [1, 2]:  # Index 1 and 2 correspond to Page 2 and Page 3
    page = doc[page_num]
    text_dict = page.get_text("dict")

    labels_to_watch = ["Product:", "Lot Number:", "Date of Manufacture:", "Expiration Date:"]
    active_labels = {}

    for block in text_dict.get("blocks", []):
        if "lines" in block:
            for line in block["lines"]:
                for span in line.get("spans", []):
                    # Clean the text thoroughly
                    text = span["text"].strip()
                    bbox = span["bbox"]  # [x0, y0, x1, y1]

                    # 1. Headers & Timestamps
                    if "Cytiva" in text and bbox[1] < 50:
                        extracted_data["basic_and_dates"].append({
                            "field": "Vendor Name", "text": text, "bbox": bbox, "page": page_num + 1
                        })
                    if "Certificate of Quality" in text:
                        extracted_data["basic_and_dates"].append({
                            "field": "Document Type", "text": text, "bbox": bbox, "page": page_num + 1
                        })
                    if "Approved On:" in text:
                        extracted_data["basic_and_dates"].append({
                            "field": "Approval Timestamp", "text": text, "bbox": bbox, "page": page_num + 1
                        })

                    # 2. Key-Value pairs with a loose proximity check
                    for label in labels_to_watch:
                        # UPGRADE: Using 'in' instead of '==' handles hidden trailing characters safely
                        if label in text:
                            active_labels[label] = bbox  # Save label coordinate

                    # Match values on the right
                    for label, label_bbox in list(active_labels.items()):
                        # We use a slight Y buffer (4 pixels) and verify it's to the right
                        if abs(bbox[1] - label_bbox[1]) < 4 and bbox[0] > label_bbox[2]:
                            # Ensure we don't accidentally capture the label itself as the value
                            if text != label:
                                extracted_data["basic_and_dates"].append({
                                    "field": label.replace(":", ""),
                                    "text": text,
                                    "bbox": bbox,
                                    "page": page_num + 1
                                })
                                del active_labels[label] # Clear anchor

In [24]:
page_3 = doc[2]
p3_dict = page_3.get_text("dict")

for block in p3_dict.get("blocks", []):
    if "lines" in block:
        for line in block["lines"]:
            for span in line.get("spans", []):
                text = span["text"].strip()
                x0, y0, x1, y1 = span["bbox"]

                # Bounding box filter: Restrict extraction to the physical grid area of the test results table
                if 360 < y0 < 500:
                    column_name = "Unknown"
                    # Classify structural columns by their hard alignment boundaries
                    if 50 <= x0 < 200:
                        column_name = "Test Description"
                    elif 200 <= x0 < 400:
                        column_name = "Test Method"
                    elif 400 <= x0 < 600:
                        column_name = "Result"

                    if text and column_name != "Unknown":
                        extracted_data["table_data"].append({
                            "column": column_name,
                            "text": text,
                            "bbox": [x0, y0, x1, y1]
                        })

In [25]:
print("================ EXTRACTED FIELDS & DATES ================")
print(json.dumps(extracted_data["basic_and_dates"][:6], indent=2))
print("\n================ EXTRACTED TABLE DATA ================")
print(json.dumps(extracted_data["table_data"][:3], indent=2))

================ EXTRACTED FIELDS & DATES ================
[
  {
    "field": "Document Type",
    "text": "Certificate of Quality",
    "bbox": [
      36.001380920410156,
      28.100793838500977,
      278.0241394042969,
      62.21213150024414
    ],
    "page": 2
  },
  {
    "field": "Lot Number",
    "text": "17242818",
    "bbox": [
      105.83869934082031,
      110.51860809326172,
      148.4272918701172,
      122.66970825195312
    ],
    "page": 2
  },
  {
    "field": "Date of Manufacture",
    "text": "20210126",
    "bbox": [
      143.39749145507812,
      132.1162567138672,
      188.27684020996094,
      144.26734924316406
    ],
    "page": 2
  },
  {
    "field": "Expiration Date",
    "text": "",
    "bbox": [
      165.95806884765625,
      153.71876525878906,
      168.20901489257812,
      165.86985778808594
    ],
    "page": 2
  },
  {
    "field": "Vendor Name",
    "text": "Cytiva",
    "bbox": [
      480.0,
      22.799999237060547,
      524.44799804687